In [ ]:
library(Seurat)
library(pheatmap)
library(RColorBrewer)
library(ComplexHeatmap)
library(tidyr)
library(ggplot2)
library(dplyr)
library(cowplot)
library(data.table)
library(harmony)
library(ggpubr)
library(ggpmisc)
set.seed(123)
library(tidyverse)
library(ggsci)
library(patchwork)
library(rhdf5)
library(future)
options(future.globals.maxSize = 50 * 1024^3)
plan(multisession, workers=16)
library(viridis)

In [ ]:
cols_celltype = c("#DC143C","#0000FF","#20B2AA","#FFA500","#9370DB","#98FB98","#E8E8E8")
names(cols_celltype) = c('EPI','PE/HYPO','TE','INTER','NR','ICM','Other')

In [ ]:
#set output
output = './01.integration_result/'

In [ ]:

###################read data###############


In [ ]:
#read self generated data
input_path = './data/'
Orangutan_bi_blastoid = readRDS(paste0(input_path,"/","Orangutan_bi_blastoids.rds"))
Orangutan_blastoid = readRDS(paste0(input_path,"/","Orangutan_blastoids.rds"))
#read public data
human_blastocyst_E5_E7 = readRDS(paste0(input_path,"/","human_blastocyst_E5_E7.rds"))
human_blastoid = readRDS(paste0(input_path,"/","human_blastoid.rds"))
macaque_blastocyst = readRDS(paste0(input_path,"/","Blastocyst_macaque.rds"))
chimpanzee_blastoid = readRDS(paste0(input_path,"/","Blastoids_chimpanzee.rds"))

In [ ]:
#Unified chimpanzee_blastoid annotation
chimpanzee_blastoid\$New_ann[which(chimpanzee_blastoid\$New_ann %in% c('TE1','TE2'))] = 'TE'
chimpanzee_blastoid\$New_ann[which(chimpanzee_blastoid\$New_ann %in% c('EPi'))] = 'EPI'
chimpanzee_blastoid\$New_ann[which(chimpanzee_blastoid\$New_ann %in% c('Hypo'))] = 'PE/HYPO'

In [ ]:

##########################check data#################


In [ ]:
data_list = c('Orangutan_bi_blastoid','Orangutan_blastoid',
              'human_blastocyst_E5_E7','human_blastoid',
              'macaque_blastocyst','chimpanzee_blastoid')
for(i in data_list){
    seuobj = get(i)
    options(repr.plot.width=7, repr.plot.height=7)
    p = DimPlot(seuobj, reduction = "umap",label = FALSE,group.by= "New_ann",cols=c(cols_celltype),pt.size =1)+coord_fixed()+ 
    theme_minimal()+theme(panel.grid.major = element_blank(), 
        panel.grid.minor = element_blank(), 
        panel.border = element_blank(), 
        axis.title = element_blank(),  
        axis.text = element_blank())+
      theme()+labs(title = "")
    print(p)
    
}

In [ ]:

########################Homologous gene conversion####################


In [ ]:
input_path = './data/'
#data from Ensembl database
geneid = read.table(paste0(input_path,'/','mart_export.txt'),sep = ',', header = TRUE, check.names = FALSE)
print(dim(geneid))
colnames(geneid) = c('Gene','GeneID','GeneID_Chimpanzee','Gene_Chimpanzee','GeneID_orangutan','Gene_orangutan','GeneID_Macaque','Gene_Macaque')
geneid = geneid[,c('Gene','Gene_Chimpanzee','Gene_orangutan','Gene_Macaque')]

build_map = function(df, species_col) {
  m = df[
    df\$Gene != "" & df[[species_col]] != "",
    c("Gene", species_col)
  ]
  m = m[!duplicated(m),]
  m = m[!duplicated(m\$Gene), ]
  m = m[!duplicated(m[[species_col]]), ]
  return(m)
}
chimp_map = build_map(geneid, "Gene_Chimpanzee")
orang_map = build_map(geneid, "Gene_orangutan")
maca_map  = build_map(geneid, "Gene_Macaque")

map_to_human_symbol = function(seurat_obj, map_df, species_col) {
  old = rownames(seurat_obj)
  idx = match(old, map_df[[species_col]])
  new = old
  new[!is.na(idx)] = map_df\$Gene[idx[!is.na(idx)]]
  print(length(new))
  print(length(unique(new)))
  keep = !duplicated(new)
  seurat_obj = seurat_obj[keep, ]
  rownames(seurat_obj) = new[keep]
  return(seurat_obj)
}

In [ ]:
chimpanzee_blastoid = map_to_human_symbol(
  chimpanzee_blastoid, chimp_map, "Gene_Chimpanzee"
)
macaque_blastocyst = map_to_human_symbol(
  macaque_blastocyst, maca_map, "Gene_Macaque"
)
Orangutan_blastoid = map_to_human_symbol(
  Orangutan_blastoid, orang_map, "Gene_orangutan"
)
Orangutan_bi_blastoid = map_to_human_symbol(
  Orangutan_bi_blastoid, orang_map, "Gene_orangutan"
)

In [ ]:
data_list = c('Orangutan_bi_blastoid','Orangutan_blastoid',
              'human_blastocyst_E5_E7','human_blastoid',
              'macaque_blastocyst','chimpanzee_blastoid')
for(i in data_list){
    seuobj = get(i)
    seuobj\$label = i
    seuobj\$cellid = rownames(seuobj@meta.data)
    seuobj@meta.data = seuobj@meta.data[,c('orig.ident','nCount_RNA','nFeature_RNA','cellid','label','New_ann')]
    assign(i,seuobj)
}

In [ ]:

###################integration#######################


In [ ]:
Data = merge(
  human_blastoid,
  y = list(
    human_blastocyst_E5_E7,
    chimpanzee_blastoid,
    macaque_blastocyst,
    Orangutan_blastoid,
    Orangutan_bi_blastoid
  ),
  add.cell.ids = c(
    "human_blastoid",
    "human_blastocyst_E5_E7",
    "chimpanzee_blastoid",
    "macaque_blastocyst",
    "Orangutan_blastoid",
    "Orangutan_bi_blastoid"
  )
)

In [ ]:
Data[["RNA"]] = JoinLayers(Data[["RNA"]])
Data[["RNA"]] = split(Data[["RNA"]], f = Data\$label)
Data = NormalizeData(Data)
Data = FindVariableFeatures(Data)
Data = ScaleData(Data)
Data = RunPCA(Data, npcs = 50)

Data = IntegrateLayers(
  object = Data,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",k.weight = 50
)

Data = FindNeighbors(Data, reduction = "integrated.cca", dims = 1:30)
Data = FindClusters(Data, resolution = 1.5, cluster.name = "cca_clusters")
Data = RunUMAP(Data, reduction = "integrated.cca", dims = 1:30, reduction.name = "umap.cca")

In [ ]:
saveRDS(Data, paste0(output,"/","CrossSpecies_CCA.rds"))